# Sapiens Q&A Bot Project Documentation

## Project Description
This project implements a Question-Answering (Q&A) bot using the 'Sapiens: A Brief History of Humankind' PDF book as its knowledge base. The bot leverages a pre-trained language model (Gemma-3-270m-it) from Hugging Face Transformers and a BM25Okapi retriever for efficient context retrieval. The process involves ingesting the PDF, splitting it into manageable chunks, tokenizing for retrieval, and then using a conversational AI model to answer user queries based *only* on the retrieved context from the book. This ensures that answers are grounded in the provided source material.

## Project Flow Diagram

```mermaid
graph TD
    A[Start] --> B(Load 'sapiens.pdf')
    B --> C{Prepare Book: Text Extraction & Chunking}
    C --> D(Data Transformation: Text to Chunks & Tokenized Corpus)
    D --> E(Initialize BM25Okapi Retriever)
    E --> F[Ask Sapiens Function: User Query]
    F --> G(Tokenize Query)
    G --> H(Retrieve Relevant Chunks using BM25Okapi)
    H --> I(Construct Prompt with Context and Query)
    I --> J(Generate Response using Gemma Model)
    J --> K(Extract Answer from Model Output)
    K --> L[Display Answer]
    L --> M[End]
```

## Step-by-Step Documentation

In [ ]:
!pip install -U "transformers>=5.5.0" accelerate bitsandbytes pypdf rank_bm25

### Step 1: Install Dependencies
This cell installs the required Python libraries: `transformers`, `accelerate`, `bitsandbytes`, `pypdf`, and `rank_bm25`.

In [2]:
import torch
import gc
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM
import pickle
import os

### Step 2: Import Libraries
This cell imports essential modules like `torch`, `gc`, `PdfReader`, `BM25Okapi`, `AutoTokenizer`, `AutoModelForCausalLM`, `pickle`, and `os` for the Q&A bot's functionality.

In [26]:
model_id = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.bfloat16,trust_remote_code=True)


def prepare_book(path):

    reader = PdfReader(path)
    text = " ".join([p.extract_text() for p in reader.pages])
    words = text.split()
    chunks = [" ".join(words[i:i+1000]) for i in range(0, len(words), 1000)]
    tokenized_corpus = [doc.lower().split() for doc in chunks]

    return chunks, BM25Okapi(tokenized_corpus)


def ask_sapiens(query, chunks, retriever):

    gc.collect()
    torch.cuda.empty_cache()

    tokenized_query = query.lower().split()
    relevant_docs = retriever.get_top_n(tokenized_query, chunks, n=3)

    print(relevant_docs)
    print('-'*25)

    context = "\n\n---\n\n".join(relevant_docs)

    messages = [{"role": "user", "content": f"Answer the question using only this context from Sapiens and make it detald:\n\n{context}\n\nQuestion: {query}"}]

    print(messages)
    print('-'*25)

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate( **inputs, max_new_tokens=500,  temperature=0.1, do_sample=False)

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return full_text.split("model\n")[-1].strip()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

### Step 3: Initialize Model and Define Helper Functions
This section sets up the core components:
1.  **Model Initialization:** Loads the `google/gemma-3-270m-it` model and its tokenizer.
2.  **`prepare_book(path)` function:** Extracts text from the PDF, chunks it into 1000-word segments, tokenizes them, and initializes a `BM25Okapi` retriever.
    - **Data Transformations:** Text is extracted from PDF, aggregated, split into words, chunked, and tokenized into a `tokenized_corpus`.
3.  **`ask_sapiens(query, chunks, retriever)` function:** Takes a query, retrieves relevant chunks using BM25, constructs a prompt, and generates an answer using the Gemma model.

In [19]:
chunks_path = 'sapiens_chunks.pkl'
retriever_path = 'sapiens_retriever.pkl'

if os.path.exists(chunks_path) and os.path.exists(retriever_path):
    print("Loading chunks and retriever from saved files...")
    with open(chunks_path, 'rb') as f:
        chunks = pickle.load(f)
    with open(retriever_path, 'rb') as f:
        retriever = pickle.load(f)
else:
    print("Processing book and initializing retriever...")
    chunks, retriever = prepare_book("sapiens.pdf")
    with open(chunks_path, 'wb') as f:
        pickle.dump(chunks, f)
    with open(retriever_path, 'wb') as f:
        pickle.dump(retriever, f)
    print("Chunks and retriever saved for future use.")

Loading chunks and retriever from saved files...


In [27]:
response = ask_sapiens("What is the 'Interbreeding Theory'?", chunks, retriever)
print(f"\nSapiens Bot: {response}")

['Humans are the outcome of blind evolutionary processes that operate without goal or purpose. Our actions are not part of some divine cosmic plan, and if planet Earth were to blow up tomorrow morning, the universe would probably keep going about its business as usual. As far as we can tell at this point, human subjectivity would not be missed. Hence any meaning that people ascribe to their lives is just a delusion. The other-worldly meanings medieval people found in their lives were no more deluded than the modern humanist, nationalist and capitalist meanings modern people find. The scientist who says her life is meaningful because she increases the store of human knowledge, the soldier who declares that his life is meaningful because he fights to defend his homeland, and the entrepreneur who finds meaning in building a new company are no less delusional than their medieval counterparts who found meaning in reading scriptures, going on a crusade or building a new cathedral. So perhaps

### Step 4: Ask a Specific Question (Interbreeding Theory)
This cell processes the PDF (or loads pre-processed data) and then uses the `ask_sapiens()` function to answer the question "What is the 'Interbreeding Theory'?", demonstrating the bot's functionality.

In [ ]:
response = ask_sapiens("Does god made us or its us who made god?", chunks, retriever)
print(f"\nSapiens Bot: {response}")

### Step 5: Ask Another Question (God's Role)
This cell asks another question, "Does god made us or its us who made god?", using the `ask_sapiens` function to show the bot's versatility.

In [ ]:
response = ask_sapiens("What was the context of Peugeot in this book?", chunks, retriever)
print(f"\nSapiens Bot: {response}")